In [ ]:
"""
Project: Sovereign Energy Drought Visualizer (The Final Trinity Edition)
Description: Merender peta Frekuensi (EDF), Durasi Max (EDD_Max), dan 
             Durasi Mean (EDD_Mean) secara berurutan + Tren Kontinu 60 Tahun.
             Dioptimasi untuk Reproducibility dan standar visualisasi Jurnal Q1.
             Revisi Layering: Garis Pantai & BORDERS dinaikkan ke zorder=4 
             agar berada di atas lapisan abu-abu muda (unviable mask).
"""

import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import pyproj
import os
import warnings

warnings.filterwarnings("ignore")

# DIREKTORI TEPAT SESUAI LOKASI LOKAL (DIPISAH CLIM & TREND)
in_dir_clim = r"C:\IPBUniversity\CoolYeah\FASTTRACK\TESIS\data\7_drouhgt-analysis\rrtm_ncld1"
in_dir_trend = r"C:\IPBUniversity\CoolYeah\FASTTRACK\TESIS\data\7_drouhgt-analysis\rrtm_ncld1\trends"
out_dir = r"C:\IPBUniversity\CoolYeah\FASTTRACK\TESIS\data\7_drouhgt-analysis\output_plots\rrtm_ncld1"
os.makedirs(out_dir, exist_ok=True)

ssps = ['ssp126', 'ssp245', 'ssp370', 'ssp585']
extent = [94, 142, -12, 9]

# --- PARAMETER MERCATOR DARI NAMELIST REGCM ---
clon, clat = 120.0, -2.5
proj_std = pyproj.Proj(f"+proj=merc +lon_0={clon} +lat_ts={clat}")
_, y_center = proj_std(clon, clat)
data_crs = ccrs.Mercator(central_longitude=clon, latitude_true_scale=clat, false_northing=-y_center)

def prep_map(ax):
    """Menyiapkan proyeksi spasial, batas pantai, dan garis negara."""
    ax.set_extent(extent, crs=ccrs.PlateCarree())
    # ⚡ REVISI ZORDER: Coastal & Borders dinaikkan ke zorder=4 agar selalu di ATAS warna abu-abu ⚡
    ax.add_feature(cfeature.COASTLINE, linewidth=0.8, edgecolor='black', zorder=4)
    ax.add_feature(cfeature.BORDERS, linewidth=0.5, edgecolor='dimgray', linestyle='--', zorder=4)

def add_label_box(ax, text, loc='bottom_left'):
    """Menambahkan kotak label skenario di sudut kanvas."""
    kwargs = dict(transform=ax.transAxes, fontsize=12, fontweight='bold', 
                  bbox=dict(boxstyle='square,pad=0.2', facecolor='white', alpha=0.85, edgecolor='gray', zorder=5))
    if loc == 'bottom_right':
        ax.text(0.96, 0.04, text, ha='right', va='bottom', **kwargs)
    elif loc == 'bottom_left':
        ax.text(0.04, 0.04, text, ha='left', va='bottom', **kwargs)

def _add_continuous_colorbar(fig, img, ax_rect, cbar_label, extend_type='max'):
    """Menambahkan colorbar horizontal pada layout bawah. (extend dibuat dinamis agar trend bisa 'both')"""
    cbar_ax = fig.add_axes(ax_rect)
    cbar = fig.colorbar(img, cax=cbar_ax, orientation='horizontal', extend=extend_type)
    
    # Memaksa ticks di colorbar menjadi bilangan bulat (Integer) murni
    cbar.locator = ticker.MaxNLocator(integer=True)
    cbar.update_ticks()
    
    cbar.set_label(cbar_label, fontsize=12, fontweight='bold')

# ⚡ KONFIGURASI THE FINAL TRINITY + TREND (DITAMBAH SOLAR SIANG) ⚡
var_config = {
    # ================= 1. FREQUENCY (EDF) =================
    'WIND_EDF':          {'vmax': 365, 'cmap': 'YlGnBu', 'trend_vmax': 10, 'label': 'Wind Drought Freq (CF < 10%) [Days/Year]'},
    'SOLAR_EDF':         {'vmax': 10,  'cmap': 'hot_r',  'trend_vmax': 2,  'label': 'Solar Drought Freq (CF < 10%) [Days/Year]'},
    'SOLAR_siang_EDF':   {'vmax': 10,  'cmap': 'hot_r',  'trend_vmax': 2,  'label': 'Solar Daylight Drought Freq (CF < 10%) [Days/Year]'},
    
    # ================= 2. WORST-CASE DURATION (EDD MAX) =================
    'WIND_EDD_Max':        {'vmax': 100, 'cmap': 'YlGnBu', 'trend_vmax': 5, 'label': 'Wind Max Drought Duration (CF < P20) [Consecutive Days]'},
    'SOLAR_EDD_Max':       {'vmax': 100, 'cmap': 'hot_r',  'trend_vmax': 5, 'label': 'Solar Max Drought Duration (CF < P20) [Consecutive Days]'},
    'SOLAR_siang_EDD_Max': {'vmax': 100, 'cmap': 'hot_r',  'trend_vmax': 5, 'label': 'Solar Daylight Max Drought Duration (CF < P20) [Consecutive Days]'},
    
    # ================= 3. AVERAGE DURATION (EDD MEAN) =================
    'WIND_EDD_Mean':        {'vmax': 40,  'cmap': 'YlGnBu', 'trend_vmax': 3, 'label': 'Wind Mean Drought Duration (CF < P20) [Days/Event]'},
    'SOLAR_EDD_Mean':       {'vmax': 30,  'cmap': 'hot_r',  'trend_vmax': 2, 'label': 'Solar Mean Drought Duration (CF < P20) [Days/Event]'},
    'SOLAR_siang_EDD_Mean': {'vmax': 30,  'cmap': 'hot_r',  'trend_vmax': 2, 'label': 'Solar Daylight Mean Drought Duration (CF < P20) [Days/Event]'}
}

def plot_drought_2x2(var_type):
    """Fungsi utama rendering matrix Climatology 2x2 dengan Unviable Masking."""
    fig, axes = plt.subplots(2, 2, figsize=(10, 5), subplot_kw={'projection': ccrs.PlateCarree()})
    axes = axes.flatten()
    
    cfg = var_config[var_type]
    vmax = cfg['vmax']
    vmin = 0 
    cmap = cfg['cmap']
    cbar_label = cfg['label']
    
    base_var = var_type.split('_ED')[0] 
    
    img = None
    for i, ssp in enumerate(ssps):
        ax = axes[i]
        prep_map(ax)
        
        fpath = os.path.join(in_dir_clim, f"Drought_Clim_{base_var}_{ssp}.nc")
        if not os.path.exists(fpath): continue
        
        with xr.open_dataset(fpath) as ds:
            data = ds[var_type].copy()
            edf_data = ds[f"{base_var}_EDF"]
            lon, lat = ds.lon, ds.lat
            
            # ⚡ EXCLUSION ZONE LOGIC ⚡
            unviable_mask = xr.where(edf_data > 360, 1, 0)
            
            # Hapus data (NaN) di daerah unviable agar colorbar tidak rusak
            data = xr.where(unviable_mask == 1, np.nan, data)
            
            img = ax.pcolormesh(lon, lat, data, transform=data_crs, 
                                cmap=cmap, vmin=vmin, vmax=vmax, shading='auto', zorder=1)
                                
            # ⚡ REVISI: Abu-abu muda ditaruh di zorder=2 (DI BISA DILIHAT DI BAWAH GARIS PANTAI zorder=4) ⚡
            ax.contourf(lon, lat, unviable_mask, levels=[0.5, 1.5], transform=data_crs,
                        colors=['lightgray'], zorder=2)
            
        add_label_box(ax, f"{ssp.upper()}", loc='bottom_left')
            
    plt.subplots_adjust(wspace=-0.035, hspace=0.02, bottom=0.15, top=0.95, left=0.05, right=0.98)
    if img: _add_continuous_colorbar(fig, img, [0.06, 0.075, 0.9, 0.04], cbar_label, extend_type='max')
    
    out_fpath = os.path.join(out_dir, f"SPATIAL_DROUGHT_CLIM_2x2_{var_type}.jpg")
    plt.savefig(out_fpath, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"✅ Rendered Climatology: {os.path.basename(out_fpath)}")

def plot_trend_2x2(var_type):
    """Fungsi Kloning Khusus Rendering Trend 60 Tahun dengan Unviable Masking."""
    fig, axes = plt.subplots(2, 2, figsize=(10, 5), subplot_kw={'projection': ccrs.PlateCarree()})
    axes = axes.flatten()
    
    cfg = var_config[var_type]
    vmax = cfg['trend_vmax']
    vmin = -vmax
    cmap = 'RdBu_r' 
    
    base_label = cfg['label'].split('[')[0].strip()
    cbar_label = f"$\\Delta$ {base_label} [Days/decade]"
    
    alpha = 0.05
    scale_factor = 10 
    
    base_var = var_type.split('_ED')[0]
    
    # ⚡ LOAD HISTORICAL BASELINE MASK ⚡
    fpath_hist = os.path.join(in_dir_clim, f"Drought_Clim_{base_var}_hist.nc")
    unviable_mask = None
    if os.path.exists(fpath_hist):
        with xr.open_dataset(fpath_hist) as ds_hist:
            edf_hist = ds_hist[f"{base_var}_EDF"]
            unviable_mask = xr.where(edf_hist > 360, 1, 0)
    
    img = None
    for i, ssp in enumerate(ssps):
        ax = axes[i]
        prep_map(ax)
        
        fpath = os.path.join(in_dir_trend, f"TREND_60YR_Drought_{base_var}_{ssp}.nc")
        if not os.path.exists(fpath): continue
        
        with xr.open_dataset(fpath) as ds:
            slope_var = f"{var_type}_slope"
            pval_var = f"{var_type}_pval"
            
            slope = ds[slope_var] * scale_factor
            pval = ds[pval_var].fillna(1.0)
            lon, lat = ds.lon, ds.lat
            
            # Masking daerah unviable dari kalkulasi trend visual
            if unviable_mask is not None:
                slope = xr.where(unviable_mask == 1, np.nan, slope)
                pval = xr.where(unviable_mask == 1, np.nan, pval)
            
            img = ax.pcolormesh(lon, lat, slope, transform=data_crs, 
                                cmap=cmap, vmin=vmin, vmax=vmax, shading='auto', zorder=1)
            
            # ⚡ REVISI: Mask abu-abu ditaruh di zorder=2 ⚡
            if unviable_mask is not None:
                ax.contourf(lon, lat, unviable_mask, levels=[0.5, 1.5], transform=data_crs,
                            colors=['lightgray'], zorder=2)

            # P-Value Hatching (Dot marker) di zorder=3
            ax.contourf(lon, lat, pval, levels=[alpha, np.inf], transform=data_crs,
                        colors='none', hatches=['...'], zorder=3)
            
        add_label_box(ax, f"Trend {ssp.upper()}", loc='bottom_left')
            
    plt.subplots_adjust(wspace=-0.035, hspace=0.02, bottom=0.15, top=0.95, left=0.05, right=0.98)
    if img: _add_continuous_colorbar(fig, img, [0.06, 0.075, 0.9, 0.04], cbar_label, extend_type='both')
    
    out_fpath = os.path.join(out_dir, f"SPATIAL_TREND_DROUGHT_2x2_{var_type}.jpg")
    plt.savefig(out_fpath, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    print(f"✅ Rendered Trend: {os.path.basename(out_fpath)}")

if __name__ == "__main__":
    print("🚀 Memulai Render Visualisasi Kekeringan Energi (Exclusion Zone Edition)...")
    for vt in var_config.keys():
        print(f"\n================ MENGGAMBAR {vt} ================")
        plot_drought_2x2(vt)
        plot_trend_2x2(vt)
    print("\n🎉 SELURUH PETA DROUGHT & TREND SELESAI DIRENDER! SIAP UNTUK PUBLIKASI.")